In [1]:
# ==========================================
# CELL 1: IMPORTS, CONFIGURATION & BACKUP
# ==========================================
import os
import shutil
import datetime
import numpy as np
import joblib
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ── DATA PATHS ───────────────────────────────────────────
# FIRST TRAINING:
# X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
# Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# RETRAINING:
# After running your combiner script, point these to:
# train_X_combined.npy and train_y_combined.npy
X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# ── MODEL PATHS ──────────────────────────────────────────
MODEL_SAVE_PATH  = r"C:\Users\kalig\OneDrive\Desktop\ovr_adaboost_model.joblib"
MODEL_BACKUP_DIR = r"C:\Users\kalig\OneDrive\Desktop\model_backups"

RANDOM_STATE = 42
os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

print("=== TRAIN / RETRAIN CONFIGURATION ===")
print(f"Feature file : {X_PATH}")
print(f"Label file   : {Y_PATH}")
print(f"Model output : {MODEL_SAVE_PATH}")

# ── BACKUP OLD MODEL ─────────────────────────────────────
if os.path.exists(MODEL_SAVE_PATH):
    timestamp   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(MODEL_BACKUP_DIR, f"ovr_adaboost_model_{timestamp}.joblib")
    shutil.copy2(MODEL_SAVE_PATH, backup_path)
    print(f" Existing model backed up to: {backup_path}")
else:
    print("No existing model found — this looks like a fresh training run.")

=== TRAIN / RETRAIN CONFIGURATION ===
Feature file : C:\Users\kalig\OneDrive\Desktop\train_X.npy
Label file   : C:\Users\kalig\OneDrive\Desktop\train_y.npy
Model output : C:\Users\kalig\OneDrive\Desktop\ovr_adaboost_model.joblib
No existing model found — this looks like a fresh training run.


In [2]:
# ==========================================
# CELL 2: LOAD, SPLIT & BALANCE DATA
# ==========================================
print("\nLoading feature matrix and labels...")
X = np.load(X_PATH)
y = np.load(Y_PATH)

print(f"Data loaded successfully! Shape: X={X.shape}, y={y.shape}")

unique, counts = np.unique(y, return_counts=True)
print(f"Full dataset class distribution: {dict(zip(unique, counts))}")

# Split first so test set remains untouched
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"\nBefore SMOTE — training distribution: {Counter(y_train)}")

smote = SMOTE(
    sampling_strategy='auto',
    random_state=RANDOM_STATE,
    k_neighbors=7
)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"After  SMOTE — training distribution: {Counter(y_train_balanced)}")

real_train_images  = X_train.shape[0]
total_train_images = X_train_balanced.shape[0]
synthetic_images   = total_train_images - real_train_images
test_images        = X_test.shape[0]

print(f"\n{'='*50}")
print("DATA SUMMARY")
print(f"{'='*50}")
print(f"Original training images   : {real_train_images}")
print(f"Synthetic images added     : {synthetic_images}")
print(f"Total training images used : {total_train_images}")
print(f"Test images                : {test_images}")
print(f"Features per image         : {X_train_balanced.shape[1]}")
print(f"{'='*50}")


Loading feature matrix and labels...
Data loaded successfully! Shape: X=(9999, 1792), y=(9999,)
Full dataset class distribution: {np.int64(1): np.int64(1172), np.int64(2): np.int64(4282), np.int64(3): np.int64(3991), np.int64(4): np.int64(554)}

Before SMOTE — training distribution: Counter({np.int64(2): 3425, np.int64(3): 3193, np.int64(1): 938, np.int64(4): 443})
After  SMOTE — training distribution: Counter({np.int64(1): 3425, np.int64(2): 3425, np.int64(3): 3425, np.int64(4): 3425})

DATA SUMMARY
Original training images   : 7999
Synthetic images added     : 5701
Total training images used : 13700
Test images                : 2000
Features per image         : 1792


In [3]:
# ==========================================
# CELL 3: BUILD & TRAIN OVR ADABOOST
# ==========================================
print("\nInitializing One-vs-Rest AdaBoost ensemble...")

base_estimator = DecisionTreeClassifier(max_depth=1)

adaboost_core = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=150,
    random_state=RANDOM_STATE
)

ovr_model = OneVsRestClassifier(adaboost_core)

print("Training 4 specialist binary models on balanced data...")
ovr_model.fit(X_train_balanced, y_train_balanced)
print(" Training complete!")


Initializing One-vs-Rest AdaBoost ensemble...
Training 4 specialist binary models on balanced data...
 Training complete!


In [4]:
# ==========================================
# CELL 4: EVALUATE & SAVE
# ==========================================
print("\nEvaluating OvR model on test set...")
y_pred = ovr_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)

print(f"\n{'='*50}")
print("PERFORMANCE REPORT")
print(f"{'='*50}")
print(f"Images model trained on    : {total_train_images}")
print(f"  ↳ Real images            : {real_train_images}")
print(f"  ↳ SMOTE synthetic        : {synthetic_images}")
print(f"Images tested on           : {test_images}")
print(f"Overall accuracy           : {acc:.4f}")
print(f"{'='*50}")

print("\nPer-class breakdown:")
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        'Tissue Density Category 1',
        'Tissue Density Category 2',
        'Tissue Density Category 3',
        'Tissue Density Category 4'
    ]
))

print("Confusion Matrix:")
header = "              " + "  ".join(f"Pred {i+1}" for i in range(4))
print(header)
for i, row in enumerate(cm):
    print(f"Actual {i+1}     " + "  ".join(f"{v:6d}" for v in row))

print(f"\nSaving model to {MODEL_SAVE_PATH}...")
joblib.dump(ovr_model, MODEL_SAVE_PATH)
print(" Done! Balanced OvR model saved.")


Evaluating OvR model on test set...

PERFORMANCE REPORT
Images model trained on    : 13700
  ↳ Real images            : 7999
  ↳ SMOTE synthetic        : 5701
Images tested on           : 2000
Overall accuracy           : 0.6495

Per-class breakdown:
                           precision    recall  f1-score   support

Tissue Density Category 1       0.51      0.76      0.61       234
Tissue Density Category 2       0.73      0.61      0.67       857
Tissue Density Category 3       0.74      0.64      0.69       798
Tissue Density Category 4       0.34      0.73      0.47       111

                 accuracy                           0.65      2000
                macro avg       0.58      0.69      0.61      2000
             weighted avg       0.69      0.65      0.66      2000

Confusion Matrix:
              Pred 1  Pred 2  Pred 3  Pred 4
Actual 1        179      55       0       0
Actual 2        169     526     153       9
Actual 3          4     135     513     146
Actual 4      